# Semantic Segmentation in Pontryagin Spaces — Colab Experiments

**Structure**
```
0. Setup & GPU check
1. Mount Google Drive
2. Clone repo & install package
3. Dataset
4. FCN backbone experiments
   4a. HPO  (optional, ~2-4 h per model)
   4b. Full training
   4c. Interpretability (SegScoreCAM + embedding energy)
   4d. Comparison
5. UNet backbone experiments
   5a. HPO  (optional)
   5b. Full training
   5c. Interpretability
   5d. Comparison
6. Download results
```

> **Before running:** set `REPO_URL`, `DATA_ZIP`, `DRIVE_DATA_ROOT`, and `DRIVE_RESULTS_ROOT` in cell **1b**.

## 0 · GPU check

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
REPO_URL         = 'https://github.com/<your-user>/003_pontriagyn_embebbed.git'

# Path to UAV_segmantation.zip already uploaded to Drive
# (or leave empty to upload manually in cell 3)
DATA_ZIP         = '/content/drive/MyDrive/UAV_segmantation.zip'

# Where the unzipped dataset will live
DRIVE_DATA_ROOT  = '/content/drive/MyDrive/UAV_segmantation'

# Where all experiment outputs will be saved (persists across sessions)
DRIVE_FCN_RESULTS  = '/content/drive/MyDrive/results/fcn'
DRIVE_UNET_RESULTS = '/content/drive/MyDrive/results/unet'
# ──────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(DRIVE_FCN_RESULTS,  exist_ok=True)
os.makedirs(DRIVE_UNET_RESULTS, exist_ok=True)
print('Paths set.')

## 2 · Clone repo & install package

In [ ]:
import os
REPO_DIR = '/content/pontryagin'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
# Install the prfe package in editable mode + any missing deps
!pip install -e . -q
!pip install scikit-optimize -q
print('Package installed.')

## 3 · Dataset

In [ ]:
import os, zipfile

if not os.path.exists(DRIVE_DATA_ROOT):
    if os.path.exists(DATA_ZIP):
        print('Unzipping dataset from Drive …')
        with zipfile.ZipFile(DATA_ZIP, 'r') as z:
            z.extractall(os.path.dirname(DRIVE_DATA_ROOT))
        print('Done.')
    else:
        # Manual upload fallback
        from google.colab import files
        print('Upload UAV_segmantation.zip manually:')
        uploaded = files.upload()
        zname = list(uploaded.keys())[0]
        with zipfile.ZipFile(zname, 'r') as z:
            z.extractall(os.path.dirname(DRIVE_DATA_ROOT))
        print('Done.')
else:
    print('Dataset already exists:', DRIVE_DATA_ROOT)

# Verify splits
for split in ('train', 'valid', 'test'):
    n = len(list(__import__('pathlib').Path(DRIVE_DATA_ROOT, split, 'images').glob('*.jpg')))
    print(f'  {split:6s}: {n} images')

## 4 · FCN backbone experiments
Models: `euclidean` | `hyperbolic` | `pontryagin`

### 4a · HPO  *(optional — skips to 4b if best_params.json exists)*

In [ ]:
# Run HPO for one model at a time.  ~2–4 h for 30 trials on a T4.
FCN_MODEL = 'pontryagin'   # change to 'hyperbolic' or 'euclidean' as needed

!python experiments/tune_seg_uav.py \
    --model        {FCN_MODEL} \
    --n-calls      30 \
    --trial-epochs 12 \
    --data-root    "{DRIVE_DATA_ROOT}" \
    --results-dir  "{DRIVE_FCN_RESULTS}"

### 4b · Full training  (60 epochs, uses best_params.json if present)

In [ ]:
# Train all three FCN models sequentially.
# To train only one: change 'all' to 'euclidean', 'hyperbolic', or 'pontryagin'.
for model in ['euclidean', 'hyperbolic', 'pontryagin']:
    !python experiments/run_seg_uav.py \
        --model            {model} \
        --load-best-params \
        --data-root        "{DRIVE_DATA_ROOT}" \
        --results-dir      "{DRIVE_FCN_RESULTS}"

### 4c · Interpretability (SegScoreCAM + embedding energy + MC Dropout)

In [ ]:
!python experiments/interpretability_analysis.py \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_FCN_RESULTS}"

In [ ]:
# Embedding energy + MC Dropout (Pontryagin only)
!python experiments/embedding_energy.py \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_FCN_RESULTS}"

### 4d · Comparison across FCN models

In [ ]:
import sys, os
sys.path.insert(0, 'experiments')
sys.path.insert(0, 'src')

# Point compare script to Drive results
os.environ['RESULTS_OVERRIDE'] = DRIVE_FCN_RESULTS

!python experiments/compare_gradcam.py

## 5 · UNet backbone experiments
Models: `vanilla` | `euclidean` | `hyperbolic` | `pontryagin`

### 5a · HPO  *(optional)*

In [ ]:
UNET_MODEL = 'pontryagin'   # change as needed

!python experiments/tune_seg_uav_unet.py \
    --model        {UNET_MODEL} \
    --n-calls      30 \
    --trial-epochs 12 \
    --data-root    "{DRIVE_DATA_ROOT}" \
    --results-dir  "{DRIVE_UNET_RESULTS}"

### 5b · Full training

In [ ]:
# Train all four UNet models sequentially.
# To train only one: replace 'all' with the model name.
!python experiments/run_seg_uav_unet.py \
    --model       all \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_UNET_RESULTS}"

### 5c · Interpretability

In [ ]:
!python experiments/interpretability_unet.py \
    --model       all \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_UNET_RESULTS}"

### 5d · Comparison across UNet models

In [ ]:
!python experiments/compare_unet.py

## 6 · Download / inspect results

In [ ]:
import json, pathlib, pandas as pd

print('=== FCN results ===')
for mt in ('euclidean', 'hyperbolic', 'pontryagin'):
    p = pathlib.Path(DRIVE_FCN_RESULTS) / mt / 'metrics.json'
    if p.exists():
        m = json.loads(p.read_text())
        print(f'  {mt:<12}  mIoU={m["macro_iou"]:.4f}  '
              f'Dice={m["macro_dice"]:.4f}  '
              f'pixel_acc={m["pixel_acc"]:.4f}')

print('\n=== UNet results ===')
for mt in ('vanilla', 'euclidean', 'hyperbolic', 'pontryagin'):
    p = pathlib.Path(DRIVE_UNET_RESULTS) / mt / 'metrics.json'
    if p.exists():
        m = json.loads(p.read_text())
        print(f'  {mt:<12}  mIoU={m["macro_iou"]:.4f}  '
              f'Dice={m["macro_dice"]:.4f}  '
              f'pixel_acc={m["pixel_acc"]:.4f}')

In [ ]:
# Zip and download all results to your local machine (optional)
import shutil
shutil.make_archive('/content/results_fcn',  'zip', DRIVE_FCN_RESULTS)
shutil.make_archive('/content/results_unet', 'zip', DRIVE_UNET_RESULTS)

from google.colab import files
files.download('/content/results_fcn.zip')
files.download('/content/results_unet.zip')